**[Course Home](../README.md) | [Python Lab](../python/05_channel_capacity_and_blahut_arimoto.ipynb) | [Prev: Ch 04: Mutual Information & Venn Geometry](04_mutual_information_and_venn_geometry.ipynb) | [Next: Ch 06: Continuous Worlds & Differential Entropy](06_differential_entropy_and_continuous_limits.ipynb)**

*Part III: Channels, Capacity & Coding Limits — Chapter 05 of 11*

---

# Chapter 5: The Limits of Communication — Channel Capacity & Blahut-Arimoto
### *Shannon's Noisy Channel Coding Theorem, Symmetric/Erasure Channels, and*
### *Alternating Convex Optimization*

---

## 1. The Core Question: Does Noise Limit Accuracy or Only Speed?

Before Claude Shannon published *"A Mathematical Theory of Communication"* in
1948, electrical engineering operated under an intuitive but fundamentally
erroneous assumption.

Engineers observed that whenever a physical medium—a copper telephone wire,
a submarine telegraph cable, or an atmospheric radio link—was subjected to
thermal noise, bits became corrupted. To reduce the probability of error $P_e$,
early telegraphers used simple repetition codes: transmit each bit three times
and take the majority vote ($0 \to 000, 1 \to 111$). To reduce error further,
repeat each bit five times, seven times, or a hundred times.

The consensus of 1940s telecommunications was clear:
*To drive the transmission error rate to zero ($P_e \to 0$), the communication
rate $R$ must inevitably drop to zero ($R \to 0$).*

### Shannon's Triumph
Shannon proved that this consensus was **flatly wrong**.

Physical noise does **not** impose an unavoidable error floor on communication.
Noise does not limit the **accuracy** of transmission; noise limits only the
**rate**!

**Shannon's Fundamental Insight:**
Every physical communication channel possesses a finite, strictly positive
number called its **Channel Capacity** $C$, measured in bits per channel use.
As long as you transmit information at a rate $R$ strictly below capacity:
$$
R < C
$$
there exist forward error-correcting codes that achieve an **arbitrarily low
probability of error**:
$$
P_e \longrightarrow 0 \quad \text{as blocklength } N \longrightarrow \infty
$$

You do *not* have to slow your transmission rate down to zero to achieve perfect
fidelity. You can transmit data at full speed—at 99.9% of channel capacity—with
arbitrarily zero errors, provided your encoding operates over sufficiently long
blocks of symbols.

---

### The Discrete Memoryless Channel (DMC) Abstraction
To study communication mathematically, Shannon formalized the **Discrete
Memoryless Channel (DMC)**:

```
  [W] -> [Encoder] -> X^N -> [CHANNEL] -> Y^N -> [Decoder] -> [W_hat]
  w in M             x in X   P(Y|X)     y in Y               w_hat
```

A DMC is formally defined by $(\mathcal{X}, P(Y \mid X), \mathcal{Y})$:
1. **Input Alphabet $\mathcal{X}$**: A finite set of transmissible symbols
   (e.g., voltages $\{+1\text{V}, -1\text{V}\}$, optical pulses $\{0, 1\}$).
2. **Output Alphabet $\mathcal{Y}$**: A finite set of received symbols.
3. **Transition Probability Matrix $P(Y \mid X)$**: The conditional
   distribution describing physical degradation. For every $x \in \mathcal{X}$
   and $y \in \mathcal{Y}$, $P(y \mid x)$ is the probability that symbol $y$ is
   received given that symbol $x$ was transmitted.

**The Memoryless Property:**
The channel is *memoryless* because physical noise affects each transmitted
symbol independently. Past inputs and outputs exert zero influence on current
transitions:
$$
P(y_1, y_2, \dots, y_N \mid x_1, x_2, \dots, x_N)
= \prod_{i=1}^N P(y_i \mid x_i)
$$

---

## 2. Channel Capacity: The Maximum Transmissible Bits

In Chapter 4, we proved that Mutual Information $I(X; Y)$ quantifies the bits of
shared knowledge between input $X$ and output $Y$:
$$
I(X; Y) = H(Y) - H(Y \mid X)
$$

In a communication system:
* The transition matrix $P(Y \mid X)$ is fixed by the laws of physics (thermal
  agitation of electrons, radio attenuation, magnetic disk grain size).
* The engineer has zero control over $P(Y \mid X)$.
* However, the engineer **has total control over the input distribution
  $P(X)$**: how frequently each symbol $x \in \mathcal{X}$ is selected for
  transmission.

### Formal Definition of Channel Capacity
The **Channel Capacity** $C$ of a discrete memoryless channel is the maximum
mutual information between channel input $X$ and channel output $Y$, maximized
over all valid input probability distributions $P(X)$:
$$
C \equiv \max_{P(X)} I(X; Y)
= \max_{P(X)} \Big[ H(Y) - H(Y \mid X) \Big]
$$
Units of $C$ are **bits per channel use** (or bits per transmission).

Because the simplex of input distributions $\Delta_{|\mathcal{X}|}$ is a compact
set and mutual information $I(X; Y)$ is a continuous, concave function of
$P(X)$ for fixed $P(Y \mid X)$, the maximum always exists and is unique in its
attained value $C$.

---

### High-Dimensional Sphere Packing: The Geometric Intuition
Why can block codes achieve zero error over a noisy wire? The answer lies in the
geometry of high-dimensional Euclidean space and the **Asymptotic Equipartition
Property (AEP)**.

Suppose we transmit sequences of blocklength $N$ (where $N \gg 1000$):

1. **Volume of the Output Space**:
   By the AEP, the set of typical received output sequences $Y^N$ has size:
   $$
   |\mathcal{A}_\epsilon^{(N)}(Y)| \approx 2^{N H(Y)}
   $$

2. **Volume of the Noise Sphere around each Codeword**:
   For any specific transmitted codeword $x^N$, the channel corrupts the bits.
   The set of conditionally typical output sequences that could have been
   generated by $x^N$ has size:
   $$
   |\mathcal{A}_\epsilon^{(N)}(Y \mid X = x^N)| \approx 2^{N H(Y \mid X)}
   $$
   Geometrically, this set forms a high-dimensional "noise sphere" centered at
   codeword $x^N$.

```
               HIGH-DIMENSIONAL SPHERE PACKING
               
      +-------------------------------------------------+
      |  Total Output Typical Space: ~ 2^{N H(Y)}      |
      |                                                 |
      |     ( Noise Sphere )       ( Noise Sphere )     |
      |     /   Codeword 1 \       /   Codeword 2 \     |
      |    |       x_1      |     |       x_2      |    |
      |     \  ~2^{NH(Y|X)}/       \  ~2^{NH(Y|X)}/     |
      |                                                 |
      |     ( Noise Sphere )       ( Noise Sphere )     |
      |     /   Codeword 3 \       /   Codeword M \     |
      |    |       x_3      |     |       x_M      |    |
      |     \  ~2^{NH(Y|X)}/       \  ~2^{NH(Y|X)}/     |
      +-------------------------------------------------+
```

3. **Maximum Disjoint Spheres Packable**:
   To decode with zero error, the noise spheres of distinct codewords must not
   overlap! If received vector $y^N$ lands inside sphere $m$, the decoder
   unambiguously announces message $m$.
   
   The maximum number of non-overlapping noise spheres $M$ that can be packed
   inside the total typical output space is:
   $$
   M \le \frac{|\mathcal{A}_\epsilon^{(N)}(Y)|}
   {|\mathcal{A}_\epsilon^{(N)}(Y \mid X)|}
   \approx \frac{2^{N H(Y)}}{2^{N H(Y \mid X)}} = 2^{N [H(Y) - H(Y \mid X)]}
   = 2^{N I(X; Y)}
   $$

4. **The Transmission Rate**:
   The transmission rate $R$ is the bits of message per channel use:
   $$
   R = \frac{\log_2 M}{N} \le H(Y) - H(Y \mid X) = I(X; Y)
   $$
   Maximizing this packing density over all possible input distributions yields
   the channel capacity:
   $$
   R \le \max_{P(X)} I(X; Y) = C
   $$

As $N \to \infty$, high-dimensional spheres develop razor-sharp boundaries
(measure concentration). Randomly generated codewords distribute their noise
spheres across the hypersphere without colliding, driving error probability
exponentially to zero!

---

## 3. Canonical Discrete Channels: Full Analytical Derivations

To build intuition, we derive the exact analytical capacity for the two most
famous channels in information theory: the **Binary Symmetric Channel (BSC)**
and the **Binary Erasure Channel (BEC)**.

---

### Channel 1: The Binary Symmetric Channel (BSC)
The BSC models physical transmission over noisy wires, radio waves, or magnetic
storage where bits are randomly flipped by additive noise.

* **Alphabet**: $\mathcal{X} = \{0, 1\}$, $\mathcal{Y} = \{0, 1\}$.
* **Transition Matrix**:
  $$
  P(Y = 0 \mid X = 0) = 1 - p, \quad P(Y = 1 \mid X = 0) = p
  $$
  $$
  P(Y = 1 \mid X = 1) = 1 - p, \quad P(Y = 0 \mid X = 1) = p
  $$
  where $p \in [0, 1]$ is the **crossover probability** (bit-flip rate).

```
         BINARY SYMMETRIC CHANNEL (BSC)
         
               1 - p
        0 --------------> 0
           \           /
            \  p   p  /
             \       /
              \     /
               \   /
        1 --------------> 1
               1 - p
```

#### First-Principles Capacity Derivation:
We expand mutual information:
$$
I(X; Y) = H(Y) - H(Y \mid X)
$$

First, evaluate the conditional entropy $H(Y \mid X)$:
$$
H(Y \mid X) = \sum_{x \in \{0, 1\}} P(X = x) H(Y \mid X = x)
$$
For $X = 0$, $Y$ is a Bernoulli random variable with parameter $p$:
$$
H(Y \mid X = 0) = -p \log_2 p - (1-p) \log_2(1-p) \equiv H_b(p)
$$
Similarly, for $X = 1$:
$$
H(Y \mid X = 1) = -p \log_2 p - (1-p) \log_2(1-p) = H_b(p)
$$
Because the conditional entropy is identical regardless of $x$:
$$
H(Y \mid X) = H_b(p) \cdot [P(X=0) + P(X=1)] = H_b(p)
$$
Notice that $H(Y \mid X)$ is a constant determined entirely by the channel
parameter $p$. It is completely independent of the input distribution $P(X)$!

Now maximize mutual information over $P(X)$:
$$
C_{\text{BSC}} = \max_{P(X)} [H(Y) - H_b(p)]
= \left[ \max_{P(X)} H(Y) \right] - H_b(p)
$$

Since $Y$ is a binary random variable ($Y \in \{0, 1\}$), its entropy is bounded
by:
$$
H(Y) \le 1.0000\text{ bit}
$$
with equality if and only if $P(Y = 0) = P(Y = 1) = 0.5$.
Because the channel is symmetric, choosing a uniform input distribution
$P(X = 0) = P(X = 1) = 0.5$ produces a perfectly uniform output distribution
$P(Y = 0) = P(Y = 1) = 0.5$.

Therefore, the maximum of $H(Y)$ is exactly $1\text{ bit}$, and we obtain:
$$
C_{\text{BSC}} = 1 - H_b(p) \quad \text{bits per channel use}
$$

#### Analysis of Edge Cases:
* **$p = 0.0$ (Clean Wire)**:
  $H_b(0) = 0 \implies C = 1 - 0 = 1.0\text{ bit/use}$.
  Every transmitted bit arrives intact.
* **$p = 0.5$ (Pure Static)**:
  $H_b(0.5) = 1 \implies C = 1 - 1 = 0.0\text{ bits/use}$.
  The channel output is purely independent random coin flips. Zero information
  can pass through the channel.
* **$p = 1.0$ (Perfect Inversion)**:
  $H_b(1.0) = 0 \implies C = 1 - 0 = 1.0\text{ bit/use}$.
  The channel flips every bit deterministically. The receiver simply flips every
  received bit back ($0 \to 1, 1 \to 0$) to recover pristine data!

---

### Channel 2: The Binary Erasure Channel (BEC)
The BEC models packet-switched communication (such as the Internet and TCP/IP)
or flash memory chips. When a packet is lost due to router buffer congestion or
checksum failure, the receiver knows the packet did not arrive. The bit is not
flipped into an erroneous value; it is **erased**.

* **Alphabet**: $\mathcal{X} = \{0, 1\}$, $\mathcal{Y} = \{0, ?, 1\}$, where
  `?` represents the erasure symbol.
* **Transition Probabilities**:
  $$
  P(Y = 0 \mid X = 0) = 1 - \epsilon, \quad P(Y = ? \mid X = 0) = \epsilon
  $$
  $$
  P(Y = 1 \mid X = 1) = 1 - \epsilon, \quad P(Y = ? \mid X = 1) = \epsilon
  $$
  where $\epsilon \in [0, 1]$ is the **erasure probability**.

```
          BINARY ERASURE CHANNEL (BEC)
          
               1 - \epsilon
        0 --------------------> 0
           \
            \  \epsilon
             \-------> ?  (Erasure)
             /------->
            /  \epsilon
           /
        1 --------------------> 1
               1 - \epsilon
```

#### First-Principles Capacity Derivation:
Let $P(X = 1) = \alpha$ and $P(X = 0) = 1 - \alpha$.
The output probabilities are:
* $P(Y = 0) = (1 - \epsilon)(1 - \alpha)$
* $P(Y = ?) = \epsilon$
* $P(Y = 1) = (1 - \epsilon)\alpha$

Let us compute the output entropy $H(Y)$:
$$
H(Y) = -P(0)\log_2 P(0) - P(?)\log_2 P(?) - P(1)\log_2 P(1)
$$
Substituting:
$$
H(Y) = -(1-\epsilon)(1-\alpha)\log_2[(1-\epsilon)(1-\alpha)]
-\epsilon \log_2 \epsilon
-(1-\epsilon)\alpha \log_2[(1-\epsilon)\alpha]
$$
Splitting logarithms: $\log_2[(1-\epsilon)u] = \log_2(1-\epsilon) + \log_2 u$:
$$
H(Y) = -\epsilon \log_2 \epsilon - (1-\epsilon)\log_2(1-\epsilon)
+ (1-\epsilon) \Big[ -(1-\alpha)\log_2(1-\alpha) - \alpha \log_2 \alpha \Big]
$$
$$
H(Y) = H_b(\epsilon) + (1 - \epsilon) H(X)
$$

Now compute conditional entropy $H(Y \mid X)$:
For both $X = 0$ and $X = 1$, the channel behaves as a binary choice between
arrival (prob $1-\epsilon$) and erasure (prob $\epsilon$).
$$
H(Y \mid X) = H_b(\epsilon)
$$

Subtracting to find mutual information:
$$
I(X; Y) = H(Y) - H(Y \mid X)
= \Big[ H_b(\epsilon) + (1-\epsilon)H(X) \Big] - H_b(\epsilon)
$$
$$
I(X; Y) = (1 - \epsilon) H(X)
$$

To maximize $I(X; Y)$, we maximize $H(X)$. Since $X \in \{0, 1\}$, the maximum
entropy is $H(X) = 1.0\text{ bit}$, achieved by uniform inputs $\alpha = 0.5$.

Therefore, the capacity of the Binary Erasure Channel is:
$$
C_{\text{BEC}} = 1 - \epsilon \quad \text{bits per channel use}
$$

---

### Deep Contrast: Bit Flips (Ambiguity) vs. Erasures (Known Loss)
Compare the capacity curves of the BSC versus the BEC:

| Error Rate | BSC Capacity ($1 - H_b(p)$) | BEC Capacity ($1 - \epsilon$) |
| :---: | :---: | :---: |
| **1%** | $1 - 0.0808 = \mathbf{0.9192}$ | $1 - 0.01 = \mathbf{0.9900}$ |
| **5%** | $1 - 0.2864 = \mathbf{0.7136}$ | $1 - 0.05 = \mathbf{0.9500}$ |
| **10%** | $1 - 0.4690 = \mathbf{0.5310}$ | $1 - 0.10 = \mathbf{0.9000}$ |
| **20%** | $1 - 0.7219 = \mathbf{0.2781}$ | $1 - 0.20 = \mathbf{0.8000}$ |

**The Mathematical Principle:**
An erasure channel preserves vastly more capacity than a flip channel with the
same error rate ($\epsilon = p$).
* In a flip channel (BSC), the receiver receives a valid bit ($0$ or $1$) but
  suffers from **ambiguity**: *did this bit flip, or did it arrive intact?*
  The receiver must expend parity check bits simply resolving which positions
  are corrupted.
* In an erasure channel (BEC), the location of data loss is **explicitly
  flagged** by the `?` token. The receiver knows with 100% certainty that every
  un-erased symbol is completely pristine!

---

## 4. The Blahut-Arimoto Algorithm (1972)

For symmetric channels like the BSC and BEC, the capacity-achieving input
distribution is trivially uniform due to geometric symmetry.

However, realistic communication channels—optical fibers with non-linear pulse
dispersion, flash memory cells with asymmetric drift, or biochemical signaling
channels—have arbitrary transition matrices $W(y \mid x)$ lacking any symmetry.
Finding the capacity $C = \max_{p(x)} I(X; Y)$ requires solving a constrained
non-linear optimization problem over the probability simplex.

In 1972, Richard Blahut (USA) and Suguru Arimoto (Japan) independently invented
an elegant, globally convergent alternating optimization algorithm.

---

### The Variational Reformulation of Capacity
Recall that mutual information can be written as:
$$
I(X; Y) = \sum_{x \in \mathcal{X}} \sum_{y \in \mathcal{Y}}
p(x) W(y \mid x) \log_2 \frac{P(x \mid y)}{p(x)}
$$
where $P(x \mid y) = \frac{p(x)W(y \mid x)}{\sum_{x'} p(x')W(y \mid x')}$ is the
true Bayesian posterior decoder.

Blahut and Arimoto introduced an arbitrary auxiliary reverse transition
distribution $Q(x \mid y)$ (a test decoder satisfying $\sum_x Q(x \mid y) = 1$
for all $y$). They proved that:
$$
I(X; Y) = \max_{Q(x \mid y)} \sum_{x, y}
p(x) W(y \mid x) \log_2 \frac{Q(x \mid y)}{p(x)}
$$
Because the difference between this objective and true mutual information is the
expected divergence
$\mathbb{E}_Y[D_{\text{KL}}(P(X \mid Y) \parallel Q(X \mid Y))] \ge 0$,
the maximum over $Q$ is achieved if and only if $Q(x \mid y)$ equals the exact
Bayes posterior!

Therefore, Channel Capacity is a **Double Maximization**:
$$
C = \max_{p(x)} \max_{Q(x \mid y)} J(p, Q)
$$
where the objective functional $J(p, Q)$ is:
$$
J(p, Q) \equiv \sum_{x \in \mathcal{X}} \sum_{y \in \mathcal{Y}}
p(x) W(y \mid x) \log_2 \frac{Q(x \mid y)}{p(x)}
$$

---

### The Alternating Iteration Steps
The double maximization is solved by alternating coordinate ascent:

#### Step 1: Optimize $Q(x \mid y)$ for fixed $p(x)$
Setting $\frac{\partial J}{\partial Q} = 0$ subject to $\sum_x Q(x \mid y) = 1$
yields the exact Bayesian posterior decoder:
$$
Q^{(t)}(x \mid y) = \frac{p^{(t)}(x) W(y \mid x)}
{\sum_{x' \in \mathcal{X}} p^{(t)}(x') W(y \mid x')}
$$

#### Step 2: Optimize $p(x)$ for fixed $Q(x \mid y)$
Setting gradient $\frac{\partial J}{\partial p} = 0$ with $\sum_x p(x) = 1$
yields an exponentiated gradient update:
Define the unnormalized weights:
$$
r^{(t+1)}(x) \equiv \exp\left(
\sum_{y \in \mathcal{Y}} W(y \mid x) \ln Q^{(t)}(x \mid y)
\right)
$$
Normalize to form the updated input distribution:
$$
p^{(t+1)}(x) = \frac{r^{(t+1)}(x)}{\sum_{x' \in \mathcal{X}} r^{(t+1)}(x')}
$$

#### Step 3: Iterate until convergence
At each iteration, compute upper and lower bounds on capacity:
$$
I_{\text{lower}}^{(t)} = J(p^{(t)}, Q^{(t)})
$$
$$
C_{\text{upper}}^{(t)} = \max_{x \in \mathcal{X}}
D_{\text{KL}}\Big( W(\cdot \mid x) \;\parallel\; P^{(t)}_Y \Big)
$$
Stop when $C_{\text{upper}}^{(t)} - I_{\text{lower}}^{(t)} < \text{tolerance}$
(e.g., $10^{-7}$).

**Convergence Guarantee:** Because $J(p, Q)$ is jointly concave in $p$ and $Q$
over the convex compact probability simplex, the Blahut-Arimoto algorithm is
mathematically guaranteed to converge monotonically to the exact global maximum!

---

## 5. Shannon's Noisy Channel Coding Theorem & Converse

We now examine Shannon's crowning mathematical achievement: the formal proof
that communication below capacity is error-free, and communication above
capacity is strictly impossible.

### Formulation of the Channel Coding Problem
* **Message**: Alice chooses a message $W$ uniformly at random from an index set
  $\mathcal{M} = \{1, 2, \dots, M\}$.
* **Encoder**: A mapping $f: \mathcal{M} \to \mathcal{X}^N$ assigns each message
  $w$ a codeword vector $x^N(w) = (x_1(w), x_2(w), \dots, x_N(w))$.
* **Code Rate**: The rate $R$ of the code measures payload bits per channel use:
  $$
  R = \frac{\log_2 M}{N} \quad \text{bits/use} \iff M = 2^{NR}
  $$
* **Channel**: Transmits $x^N$ and outputs $y^N$ according to $P(y^N \mid x^N)$.
* **Decoder**: A mapping $g: \mathcal{Y}^N \to \mathcal{M}$ assigns received
  vector $y^N$ an estimated message $\hat{W}$.
* **Average Error Probability**:
  $$
  P_e^{(N)} = \frac{1}{M} \sum_{w=1}^M P\Big( g(Y^N) \ne w \;\Big|\; W = w \Big)
  $$

---

### Theorem 1: The Channel Coding Theorem (Shannon 1948)
**Forward Theorem (Achievability):**
For any channel with capacity $C$, and for any transmission rate $R < C$, there
exists a sequence of $(2^{NR}, N)$ codes such that the maximal error
probability satisfies:
$$
\lim_{N \to \infty} P_e^{(N)} = 0
$$

#### Shannon's Random Coding Proof Strategy:
To prove an optimal code exists, Shannon introduced the revolutionary technique
of **Random Coding**:
1. Do not attempt to construct a specific handcrafted code.
2. Instead, generate all $2^{NR}$ codewords at random, drawing each symbol
   i.i.d. from the capacity-achieving distribution $P^*(X)$.
3. Use a **Joint Typicality Decoder**: Given received sequence $y^N$, search for
   a codeword $x^N(w)$ such that $(x^N(w), y^N)$ are jointly typical.
4. Two types of error can occur:
   * **Error Type 1**: The true transmitted codeword $x^N(w)$ is not jointly
     typical with $y^N$. By the Law of Large Numbers, this probability vanishes
     as $N \to \infty$.
   * **Error Type 2**: Some other un-transmitted codeword $x^N(w')$ ($w' \ne w$)
     is accidentally jointly typical with $y^N$.
     By the Joint AEP, the probability that an independent random codeword is
     jointly typical with $y^N$ is approximately $2^{-N I(X; Y)}$.
     By the Union Bound over all $2^{NR} - 1$ competing codewords:
     $$
     P(\text{Error Type 2}) \le 2^{NR} \cdot 2^{-N I(X; Y)} = 2^{-N (C - R)}
     $$
5. If $R < C$, the exponent $-(C - R)$ is strictly negative!
   As blocklength $N \to \infty$:
   $$
   P_e^{(N)} \le 2^{-N(C - R)} \longrightarrow 0 \quad (\text{exponentially})
   $$
Because the *average* error probability over the random ensemble of codebooks
approaches zero, there must exist at least one specific, deterministic codebook
whose error probability is at least as good as the average!

---

### Theorem 2: The Strong Converse
Can an engineer push transmission rate $R$ above channel capacity $C$ by
tolerating a tiny 1% error rate?

**The Strong Converse (Wolfowitz 1957):**
If $R > C$, then for *any* sequence of codes:
$$
\lim_{N \to \infty} P_e^{(N)} = 1.0000
$$
Communication at rates above capacity does not simply degrade; it fails
**completely and catastrophically**. Every single received message becomes
corrupted as blocklength grows. Channel capacity $C$ is an impenetrable physical
wall.

---

### Proof of Weak Converse via Fano's Inequality
We prove that reliable communication ($P_e \to 0$) requires $R \le C$ using
**Fano's Inequality**.

Recall Fano's Inequality: if message $W$ is estimated by $\hat{W} = g(Y^N)$ with
error probability $P_e$, the remaining uncertainty in $W$ is bounded by:
$$
H(W \mid Y^N) \le 1 + P_e \log_2(M - 1) \le 1 + P_e N R
$$

Now expand the entropy of the transmitted message $W$:
$$
N R = H(W) = H(W \mid Y^N) + I(W; Y^N)
$$
Apply Fano's Inequality:
$$
N R \le 1 + P_e N R + I(W; Y^N)
$$

By the Data Processing Inequality, because $W \to X^N \to Y^N$ forms a Markov
chain:
$$
I(W; Y^N) \le I(X^N; Y^N)
$$
For a memoryless channel, the mutual information of a length-$N$ block is
bounded by $N$ times single-letter capacity:
$$
I(X^N; Y^N) \le \sum_{i=1}^N I(X_i; Y_i) \le N C
$$

Substituting this into the inequality:
$$
N R \le 1 + P_e N R + N C
$$
Divide both sides by $N$:
$$
R \le C + P_e R + \frac{1}{N}
$$

If we demand reliable communication such that $P_e \to 0$ as $N \to \infty$:
$$
R \le C
$$
$\blacksquare$

---

## 6. Applied Engineering Contexts

### 1. Modern Error-Correcting Codes: Approaching the Shannon Limit
Shannon's 1948 proof was non-constructive: it proved that capacity-achieving
random codes *exist*, but decoding a truly random codebook of size $2^{NR}$
requires comparing against billions of vectors, which is computationally
impossible ($O(2^{NR})$ complexity).

For 50 years, finding computationally efficient, capacity-achieving codes was
the Holy Grail of coding theory. Today, that quest is complete:

* **Low-Density Parity-Check (LDPC) Codes**:
  Invented by Robert Gallager in his 1963 MIT PhD thesis and forgotten until
  rediscovered by David MacKay in 1996. LDPC codes use sparse bipartite graphs
  (Tanner graphs) decoded via iterative belief propagation (the sum-product
  algorithm).
  * In 2001, Chung et al. engineered an irregular LDPC code operating within
    **0.0045 dB** of the Shannon capacity limit!
  * **Production Deployment**: Standardized in Wi-Fi 6/7 (802.11ax/be), 5G NR
    data channels (3GPP eMBB), and satellite broadcast (DVB-S2X).
* **Polar Codes**:
  Invented by Erdal Arıkan in 2009. Polar codes use recursive Kronecker
  polarization matrices to polarize $N$ independent channels into extremal
  sub-channels that are either purely clean ($C=1$) or purely noisy ($C=0$).
  * The first mathematically proven explicit codes to achieve capacity for
    arbitrary DMCs with low complexity ($O(N \log N)$ successive cancellation
    list decoding).
  * **Production Deployment**: Standardized as the official error-correcting
    code for all 5G NR control channels (PBCH, PDCCH, PUCCH).

---

### 2. Erasure Coding in Cloud Distributed Storage
In massive distributed storage systems (Google Colossus, Apache Hadoop HDFS,
Ceph, AWS S3), disk drives fail continuously.

* **Storage Node Failure as a BEC**:
  When a hard drive or flash controller crashes, the failure is an **erasure**
  (not a bit flip): the filesystem knows precisely which drive failed because
  the storage daemon stops responding or reports SATA read errors.
* **The Cost of Naive Replication**:
  Early datacenters used 3-way replication (storing 3 identical copies of every
  chunk), incurring a **200% storage overhead** (effective rate $R = 1/3$).
* **Reed-Solomon Erasure Coding**:
  Modern distributed filesystems partition data into $k$ data chunks and compute
  $m$ parity chunks using Reed-Solomon $(k + m)$ codes over Galois fields.
  * For example, Google Colossus and Ceph commonly deploy $(k=6, m=3)$ or
    $(k=8, m=4)$ configurations.
  * An $(8 + 4)$ code tolerates the simultaneous loss of any 4 disk drives with
    only **50% storage overhead** (effective rate $R = 8/12 = 0.667$).
  * Operating at the BEC capacity limit saves cloud providers billions of
    dollars in disk purchases and datacenter power.

---

### 3. Deep Space Communications & The Extreme Power Limit
The NASA Deep Space Network (DSN) communicates with Voyager 1 and Voyager 2,
currently over 15 billion miles from Earth.

* **Extreme Attenuation**:
  Radio signals from Voyager arrive at Earth with power measured in attowatts
  ($10^{-18}\text{ W}$), far below ambient cosmic thermal noise.
* **Operating Near the Ultimate Shannon Limit**:
  In power-limited Gaussian channels, as bandwidth $W \to \infty$, the capacity
  approaches the ultimate Shannon limit:
  $$
  \frac{E_b}{N_0} \ge \ln(2) = -1.59\text{ dB}
  $$
  NASA deep space probes transmit concatenated convolutional and LDPC codes
  operating within $0.5\text{ dB}$ of this ultimate theoretical boundary.
  Without Shannon's channel capacity theorem, interplanetary exploration and
  deep space imagery would be physically impossible.

---

**[Course Home](../README.md) | [Python Lab](../python/05_channel_capacity_and_blahut_arimoto.ipynb) | [Prev: Ch 04: Mutual Information & Venn Geometry](04_mutual_information_and_venn_geometry.ipynb) | [Next: Ch 06: Continuous Worlds & Differential Entropy](06_differential_entropy_and_continuous_limits.ipynb)**

